In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [20]:
file_path = "D:/clv/data/processed/cleaned_retail.csv"

df = pd.read_csv(
    file_path,
    parse_dates=["invoice_date"]
)

print("Dataset shape:", df.shape)

Dataset shape: (1033036, 13)


In [21]:
df.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_cancellation,is_return,is_sale,line_value,sales_value
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,False,False,True,83.40,83.40
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,False,False,True,81.00,81.00
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,False,False,True,81.00,81.00
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,False,False,True,100.80,100.80
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,False,False,True,30.00,30.00


In [22]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 13 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   invoice          1033036 non-null  str           
 1   stock_code       1033036 non-null  str           
 2   description      1028761 non-null  str           
 3   quantity         1033036 non-null  int64         
 4   invoice_date     1033036 non-null  datetime64[us]
 5   price            1033036 non-null  float64       
 6   customer_id      797885 non-null   float64       
 7   country          1033036 non-null  str           
 8   is_cancellation  1033036 non-null  bool          
 9   is_return        1033036 non-null  bool          
 10  is_sale          1033036 non-null  bool          
 11  line_value       1033036 non-null  float64       
 12  sales_value      1033036 non-null  float64       
dtypes: bool(3), datetime64[us](1), float64(4), int64(1), str(4)
memory u

In [23]:
required_columns = [
    "invoice",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "price",
    "customer_id",
    "country",
    "is_cancellation",
    "is_return",
    "is_sale",
    "line_value",
    "sales_value"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns from cleaned dataset: {missing_columns}"
    )

print("✓ Input validation successful")

✓ Input validation successful


In [24]:
print("Rows:", len(df))
print("Unique invoices:", df["invoice"].nunique())
print("Unique products:", df["stock_code"].nunique())
print("Unique customers:", df["customer_id"].nunique())
print("Date range:")
print(df["invoice_date"].min())
print(df["invoice_date"].max())

Rows: 1033036
Unique invoices: 53628
Unique products: 5305
Unique customers: 5942
Date range:
2009-12-01 07:45:00
2011-12-09 12:50:00


In [25]:
customer_df = df[
    df["customer_id"].notna()
].copy()

print("Customer-analysis rows:", len(customer_df))
print(
    "Identifiable customers:",
    customer_df["customer_id"].nunique()
)

Customer-analysis rows: 797885
Identifiable customers: 5942


In [26]:
sales_df = customer_df[
    customer_df["is_sale"]
].copy()

returns_df = customer_df[
    customer_df["is_return"]
].copy()

print("Sales rows:", len(sales_df))
print("Return rows:", len(returns_df))

print(
    "Sales revenue:",
    sales_df["sales_value"].sum()
)

print(
    "Return value:",
    returns_df["line_value"].sum()
)

Sales rows: 779495
Return rows: 18390
Sales revenue: 17374804.268
Return value: -1084812.98


In [27]:
analysis_date = (
    sales_df["invoice_date"].max()
    + pd.Timedelta(days=1)
)

print("Analysis date:", analysis_date)

Analysis date: 2011-12-10 12:50:00


In [28]:
customer_features = (
    sales_df
    .groupby("customer_id")
    .agg(
        total_orders=("invoice", "nunique"),
        total_items=("quantity", "sum"),
        total_revenue=("sales_value", "sum"),
        first_purchase=("invoice_date", "min"),
        last_purchase=("invoice_date", "max")
    )
    .reset_index()
)

In [29]:
#recency
customer_features["recency"] = (
    analysis_date
    - customer_features["last_purchase"]
).dt.days

#frequency
customer_features["frequency"] = (
    customer_features["total_orders"]
)
#monetary
customer_features["monetary"] = (
    customer_features["total_revenue"]
)
customer_features[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary"
    ]
].head()


,customer_id,recency,frequency,monetary
0,"12,346.00",326,12,"77,556.46"
1,"12,347.00",2,8,"4,921.53"
2,"12,348.00",75,5,"2,019.40"
3,"12,349.00",19,4,"4,428.69"
4,"12,350.00",310,1,334.40


In [30]:
customer_features[
    [
        "recency",
        "frequency",
        "monetary"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
recency,"5,881.00",201.46,209.47,1.00,26.00,96.00,380.00,739.00
frequency,"5,881.00",6.29,13.01,1.00,1.00,3.00,7.00,398.00
monetary,"5,881.00","2,954.40","14,437.32",0.00,341.90,865.60,"2,247.72","580,987.04"


In [31]:
customer_features["avg_order_value"] = (
    customer_features["monetary"]
    / customer_features["frequency"]
)

In [32]:
customer_features["avg_order_value"] = (
    customer_features["monetary"]
    / customer_features["frequency"]
)
customer_features["avg_items_per_order"] = (
    customer_features["total_items"]
    / customer_features["frequency"]
)
unique_products = (
    sales_df
    .groupby("customer_id")["stock_code"]
    .nunique()
    .rename("unique_products")
)

customer_features = customer_features.merge(
    unique_products,
    on="customer_id",
    how="left"
)


In [33]:
active_months = (
    sales_df
    .assign(
        year_month=sales_df["invoice_date"].dt.to_period("M")
    )
    .groupby("customer_id")["year_month"]
    .nunique()
    .rename("active_months")
)

customer_features = customer_features.merge(
    active_months,
    on="customer_id",
    how="left"
)

customer_features["customer_lifetime_days"] = (
    customer_features["last_purchase"]
    - customer_features["first_purchase"]
).dt.days




In [34]:
sales_sorted = sales_df.sort_values(
    ["customer_id", "invoice_date"]
).copy()
sales_sorted["days_since_previous"] = (
    sales_sorted
    .groupby("customer_id")["invoice_date"]
    .diff()
    .dt.days
)
purchase_interval = (
    sales_sorted
    .groupby("customer_id")["days_since_previous"]
    .mean()
    .rename("avg_purchase_interval_days")
)

customer_features = customer_features.merge(
    purchase_interval,
    on="customer_id",
    how="left"
)

sales_df["day_of_week"] = (
    sales_df["invoice_date"].dt.dayofweek
)

sales_df["is_weekend"] = (
    sales_df["day_of_week"] >= 5
)
weekend_ratio = (
    sales_df
    .groupby("customer_id")["is_weekend"]
    .mean()
    .rename("weekend_purchase_ratio")
)

customer_features = customer_features.merge(
    weekend_ratio,
    on="customer_id",
    how="left"
)




In [35]:
return_orders = (
    returns_df
    .groupby("customer_id")["invoice"]
    .nunique()
    .rename("return_orders")
)
returned_items = (
    returns_df
    .groupby("customer_id")["quantity"]
    .sum()
    .abs()
    .rename("returned_items")
)

customer_features = customer_features.merge(
    return_orders,
    on="customer_id",
    how="left"
)

customer_features = customer_features.merge(
    returned_items,
    on="customer_id",
    how="left"
)

customer_features[
    ["return_orders", "returned_items"]
] = customer_features[
    ["return_orders", "returned_items"]
].fillna(0)

In [36]:
customer_features["return_order_rate"] = (
    customer_features["return_orders"]
    / customer_features["frequency"]
)

customer_features["return_item_rate"] = (
    customer_features["returned_items"]
    /
    (
        customer_features["total_items"]
        + customer_features["returned_items"]
    )
)

In [37]:
customer_features["R_score"] = pd.qcut(
    customer_features["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1],
    duplicates="drop"
).astype(int)

customer_features["F_score"] = pd.qcut(
    customer_features["frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

customer_features["M_score"] = pd.qcut(
    customer_features["monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

In [38]:
customer_features["RFM_score"] = (
    customer_features["R_score"].astype(str)
    + customer_features["F_score"].astype(str)
    + customer_features["M_score"].astype(str)
)

In [44]:
# RFM scores

# Recency: lower is better
customer_features["R_score"] = pd.qcut(
    customer_features["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1],
    duplicates="drop"
).astype(int)

# Frequency: higher is better
customer_features["F_score"] = pd.qcut(
    customer_features["frequency"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# Monetary: higher is better
customer_features["M_score"] = pd.qcut(
    customer_features["monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)
customer_features["RFM_score"] = (
    customer_features["R_score"].astype(str)
    + customer_features["F_score"].astype(str)
    + customer_features["M_score"].astype(str)
)
def assign_rfm_segment(row):

    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    elif r >= 3 and f >= 4:
        return "Loyal Customers"

    elif r >= 4 and f <= 3:
        return "Potential Loyalists"

    elif r <= 2 and f >= 3 and m >= 3:
        return "At Risk"

    elif r <= 2 and f <= 2:
        return "Lost"

    else:
        return "Regular Customers"
    
customer_features["rfm_segment"] = customer_features.apply(
    assign_rfm_segment,
    axis=1
)



rfm_summary = (
    customer_features
    .groupby("rfm_segment")
    .agg(
        customers=("customer_id", "nunique"),
        revenue=("monetary", "sum"),
        avg_revenue=("monetary", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_recency=("recency", "mean")
    )
    .reset_index()
)

rfm_summary["revenue_share"] = (
    rfm_summary["revenue"]
    / rfm_summary["revenue"].sum()
)

In [40]:
feature_missing = pd.DataFrame({
    "Missing Count": customer_features.isna().sum(),
    "Missing %": (
        customer_features.isna().mean() * 100
    ).round(2)
})

feature_missing.sort_values(
    "Missing %",
    ascending=False
)
customer_features.describe().T


,count,mean,min,25%,50%,75%,max,std
customer_id,"5,881.00","15,314.67","12,346.00","13,833.00","15,313.00","16,797.00","18,287.00","1,715.43"
total_orders,"5,881.00",6.29,1.00,1.00,3.00,7.00,398.00,13.01
total_items,"5,881.00","1,790.29",1.00,186.00,480.00,"1,350.00","367,833.00","8,882.01"
total_revenue,"5,881.00","2,954.40",0.00,341.90,865.60,"2,247.72","580,987.04","14,437.32"
first_purchase,5881,2010-08-22 07:16:55.031457,2009-12-01 07:45:00,2010-02-09 14:17:00,2010-06-27 13:29:00,2011-01-30 14:36:00,2011-12-09 12:16:00,NaN
last_purchase,5881,2011-05-22 13:18:31.749702,2009-12-01 09:55:00,2010-11-25 10:11:00,2011-09-05 11:52:00,2011-11-14 11:33:00,2011-12-09 12:50:00,NaN
recency,"5,881.00",201.46,1.00,26.00,96.00,380.00,739.00,209.47
frequency,"5,881.00",6.29,1.00,1.00,3.00,7.00,398.00,13.01
monetary,"5,881.00","2,954.40",0.00,341.90,865.60,"2,247.72","580,987.04","14,437.32"
avg_order_value,"5,881.00",384.98,0.00,176.63,279.13,414.90,"84,236.25","1,214.00"


In [45]:
customer_features.to_csv(
    "D:/clv/data/processed/customer_features.csv",
    index=False
)

rfm_summary.to_csv(
    "D:/clv/data/processed/rfm_segment_summary.csv",
    index=False
)

In [46]:
print(customer_features.columns.tolist())

['customer_id', 'total_orders', 'total_items', 'total_revenue', 'first_purchase', 'last_purchase', 'recency', 'frequency', 'monetary', 'avg_order_value', 'avg_items_per_order', 'unique_products', 'active_months', 'customer_lifetime_days', 'avg_purchase_interval_days', 'weekend_purchase_ratio', 'return_orders', 'returned_items', 'return_order_rate', 'return_item_rate', 'R_score', 'F_score', 'M_score', 'RFM_score', 'rfm_segment']
